# Praktikum 1.2 — Mempelajari Penggunaan Driver Neo4j


## 1. Impor library dan siapkan koneksi

In [11]:
from neo4j import GraphDatabase, RoutingControl
from getpass import getpass

# URI sesuai contoh pada modul
URI = "neo4j://localhost:7687"
USERNAME = "neo4j"

# Masukkan password database Neo4j saat diminta.
# Password tidak ditulis langsung di notebook agar tidak tersimpan di file.
PASSWORD = getpass("Masukkan password Neo4j: ")

# Nama database sesuai contoh pada modul
DATABASE = "neo4j"

## 2. Membuat fungsi `add_friend`

Fungsi ini membuat node untuk dua orang (jika belum ada), lalu membuat relasi `KNOWS` dari orang pertama ke temannya. `MERGE` digunakan agar data yang sama tidak dibuat berulang kali.

In [12]:
def add_friend(driver, name, friend_name):
    driver.execute_query(
        "MERGE (a:Person {name: $name}) "
        "MERGE (friend:Person {name: $friend_name}) "
        "MERGE (a)-[:KNOWS]->(friend)",
        name=name,
        friend_name=friend_name,
        database_=DATABASE,
    )

## 3. Membuat fungsi `print_friends`

Fungsi berikut mencari semua node `Person` yang terhubung dari orang yang dipilih melalui relasi `KNOWS`, kemudian mencetak nama teman secara alfabetis.

In [13]:
def print_friends(driver, name):
    records, summary, keys = driver.execute_query(
        "MATCH (a:Person)-[:KNOWS]->(friend) "
        "WHERE a.name = $name "
        "RETURN friend.name AS friend_name "
        "ORDER BY friend.name",
        name=name,
        database_=DATABASE,
        routing_=RoutingControl.READ,
    )

    if not records:
        print(f"Belum ada teman yang tercatat untuk {name}.")
        return

    for record in records:
        print(record["friend_name"])

## 4. Jalankan contoh pada modul

In [10]:
with GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD)) as driver:
    # Memeriksa apakah koneksi berhasil
    driver.verify_connectivity()
    print("Koneksi ke Neo4j berhasil.")

    # Membuat relasi pertemanan sesuai contoh modul
    add_friend(driver, "Arthur", "Guinevere")
    add_friend(driver, "Arthur", "Lancelot")
    add_friend(driver, "Arthur", "Merlin")

    # Menampilkan teman Arthur
    print("Teman Arthur:")
    print_friends(driver, "Arthur")

Koneksi ke Neo4j berhasil.
Teman Arthur:
Guinevere
Lancelot
Merlin
